# Discourse Regimes Core (2020-2022)

This notebook builds on the cleaned WSB corpus and sentiment outputs to identify discourse regimes (clusters), quantify emotional alignment and linguistic convergence, and track how these metrics co-vary over time during 2020–2022. The goal is descriptive: map regime dominance and diversity shifts around major market events without making causal claims.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.appName("wsb_regimes_core").getOrCreate()

SUBMISSIONS_CLEAN_PATH = "/project/macss/amritap1/redditproject/data/processed/submissions_clean_parquet"
COMMENTS_CLEAN_PATH = "/project/macss/amritap1/redditproject/data/processed/comments_clean_parquet"

SUBMISSIONS_SENT_PATH = "/project/macss/amritap1/redditproject/data/processed/submissions_with_sentiment"
COMMENTS_SENT_PATH = "/project/macss/amritap1/redditproject/data/processed/comments_with_sentiment"

print("Spark ready.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 06:05:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready.


In [2]:
submissions = spark.read.parquet(SUBMISSIONS_CLEAN_PATH)
comments = spark.read.parquet(COMMENTS_CLEAN_PATH)

submissions_sent = spark.read.parquet(SUBMISSIONS_SENT_PATH)
comments_sent = spark.read.parquet(COMMENTS_SENT_PATH)

# Ensure date type
submissions = submissions.withColumn("date", F.to_date("date"))
comments = comments.withColumn("date", F.to_date("date"))
submissions_sent = submissions_sent.withColumn("date", F.to_date("date"))
comments_sent = comments_sent.withColumn("date", F.to_date("date"))

# print("Submissions:", submissions.count())
# print("Comments:", comments.count())

In [3]:
t0, t1 = "2020-01-01", "2022-12-31"

subs_core = (
    submissions
    .filter(F.col("date").between(t0, t1))
    .select("id", "date", "author", "text")
    .dropna(subset=["text"])
)

coms_core = (
    comments
    .filter(F.col("date").between(t0, t1))
    .select("id", "date", "author", "text")
    .dropna(subset=["text"])
)

docs = (
    subs_core.select(F.col("id").alias("doc_id"), "date", "author", "text")
    .unionByName(
        coms_core.select(F.col("id").alias("doc_id"), "date", "author", "text")
    )
)

print("Docs:", docs.count())

[Stage 4:=====================================================>   (63 + 4) / 67]

Docs: 55799478


In [4]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, PCA
from pyspark.sql import functions as F

tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens",
    pattern=r"(?u)(?:\$\w+|[A-Za-z][\w'-]{2,})",
    gaps=False,
    toLowercase=True
)
tok = tokenizer.transform(docs)

stopwords = StopWordsRemover.loadDefaultStopWords("english")
remover = StopWordsRemover(inputCol="tokens", outputCol="tokens_clean", stopWords=stopwords)
clean = remover.transform(tok).filter(F.size("tokens_clean") >= 3)

# Smaller feature space
htf = HashingTF(inputCol="tokens_clean", outputCol="termFreq", numFeatures=1000)
tf = htf.transform(clean)

idf = IDF(inputCol="termFreq", outputCol="features")
idf_model = idf.fit(tf)
tfidf = idf_model.transform(tf)

# Fit PCA on a smaller sample, no cache
sample = tfidf.sample(fraction=0.01, seed=42)
pca = PCA(k=30, inputCol="features", outputCol="pcaFeatures")
pca_model = pca.fit(sample)

pca_df = pca_model.transform(tfidf).select("doc_id", "date", "pcaFeatures")
print("HashingTF + PCA ready.")

26/04/20 06:15:34 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/20 06:15:34 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/04/20 06:21:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


HashingTF + PCA ready.


In [5]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark import StorageLevel

# one small sample, persisted to disk to avoid memory pressure
sample = pca_df.sample(fraction=0.01, seed=42).persist(StorageLevel.DISK_ONLY)

evaluator = ClusteringEvaluator(
    featuresCol="pcaFeatures",
    predictionCol="cluster",
    metricName="silhouette"
)

for k in [3, 4, 5, 6]:
    km = KMeans(k=k, seed=42, featuresCol="pcaFeatures", predictionCol="cluster")
    model = km.fit(sample)
    pred = model.transform(sample)
    score = evaluator.evaluate(pred)
    print(f"k={k}, silhouette={score:.4f}")

k=3, silhouette=0.9999
k=4, silhouette=0.9845
k=5, silhouette=0.9851
k=6, silhouette=0.9864


In [6]:
from pyspark.ml.clustering import KMeans

def fit_k_and_show_sizes(k):
    km = KMeans(k=k, seed=42, featuresCol="pcaFeatures", predictionCol="cluster")
    model = km.fit(pca_df)
    clustered_k = model.transform(pca_df).select("doc_id", "date", "cluster")
    print(f"\n=== k={k} cluster sizes ===")
    clustered_k.groupBy("cluster").count().orderBy("cluster").show()
    return clustered_k

clustered_3 = fit_k_and_show_sizes(3)
clustered_4 = fit_k_and_show_sizes(4)
clustered_5 = fit_k_and_show_sizes(5)

26/04/20 06:40:33 WARN MemoryStore: Not enough space to cache rdd_395_14 in memory! (computed 5.3 MiB so far)
26/04/20 06:40:33 WARN BlockManager: Persisting block rdd_395_14 to disk instead.
26/04/20 06:40:34 WARN MemoryStore: Not enough space to cache rdd_395_11 in memory! (computed 12.0 MiB so far)
26/04/20 06:40:34 WARN BlockManager: Persisting block rdd_395_11 to disk instead.
26/04/20 06:40:36 WARN MemoryStore: Not enough space to cache rdd_395_13 in memory! (computed 18.0 MiB so far)
26/04/20 06:40:36 WARN BlockManager: Persisting block rdd_395_13 to disk instead.
26/04/20 06:40:38 WARN MemoryStore: Not enough space to cache rdd_395_15 in memory! (computed 8.0 MiB so far)
26/04/20 06:40:38 WARN BlockManager: Persisting block rdd_395_15 to disk instead.
26/04/20 06:40:39 WARN MemoryStore: Not enough space to cache rdd_395_10 in memory! (computed 27.0 MiB so far)
26/04/20 06:40:39 WARN BlockManager: Persisting block rdd_395_10 to disk instead.
26/04/20 06:40:42 WARN MemoryStore: N


=== k=3 cluster sizes ===


+-------+--------+
|cluster|   count|
+-------+--------+
|      0|42356235|
|      1|  240135|
|      2|     103|
+-------+--------+



26/04/20 07:10:23 WARN MemoryStore: Not enough space to cache rdd_469_10 in memory! (computed 12.0 MiB so far)
26/04/20 07:10:23 WARN BlockManager: Persisting block rdd_469_10 to disk instead.
26/04/20 07:10:24 WARN MemoryStore: Not enough space to cache rdd_469_12 in memory! (computed 12.0 MiB so far)
26/04/20 07:10:24 WARN BlockManager: Persisting block rdd_469_12 to disk instead.
26/04/20 07:10:26 WARN MemoryStore: Not enough space to cache rdd_469_13 in memory! (computed 18.0 MiB so far)
26/04/20 07:10:26 WARN BlockManager: Persisting block rdd_469_13 to disk instead.
26/04/20 07:10:28 WARN MemoryStore: Not enough space to cache rdd_469_16 in memory! (computed 8.0 MiB so far)
26/04/20 07:10:28 WARN BlockManager: Persisting block rdd_469_16 to disk instead.
26/04/20 07:10:29 WARN MemoryStore: Not enough space to cache rdd_469_11 in memory! (computed 27.0 MiB so far)
26/04/20 07:10:29 WARN BlockManager: Persisting block rdd_469_11 to disk instead.
26/04/20 07:10:32 WARN MemoryStore: 


=== k=4 cluster sizes ===


+-------+--------+
|cluster|   count|
+-------+--------+
|      0|42356227|
|      1|  240109|
|      2|      33|
|      3|     104|
+-------+--------+



26/04/20 07:40:29 WARN MemoryStore: Not enough space to cache rdd_545_14 in memory! (computed 5.3 MiB so far)
26/04/20 07:40:29 WARN BlockManager: Persisting block rdd_545_14 to disk instead.
26/04/20 07:40:29 WARN MemoryStore: Not enough space to cache rdd_545_11 in memory! (computed 12.0 MiB so far)
26/04/20 07:40:29 WARN BlockManager: Persisting block rdd_545_11 to disk instead.
26/04/20 07:40:31 WARN MemoryStore: Not enough space to cache rdd_545_13 in memory! (computed 18.0 MiB so far)
26/04/20 07:40:31 WARN BlockManager: Persisting block rdd_545_13 to disk instead.
26/04/20 07:40:34 WARN MemoryStore: Not enough space to cache rdd_545_16 in memory! (computed 8.0 MiB so far)
26/04/20 07:40:34 WARN BlockManager: Persisting block rdd_545_16 to disk instead.
26/04/20 07:40:35 WARN MemoryStore: Not enough space to cache rdd_545_10 in memory! (computed 27.0 MiB so far)
26/04/20 07:40:35 WARN BlockManager: Persisting block rdd_545_10 to disk instead.
26/04/20 07:40:36 WARN MemoryStore: N


=== k=5 cluster sizes ===


[Stage 250:======================================================>(66 + 1) / 67]

+-------+--------+
|cluster|   count|
+-------+--------+
|      0|41444120|
|      1|  237449|
|      2|      33|
|      3|     104|
|      4|  914767|
+-------+--------+

